# Next-Day PM2.5 Forecast: Milestone 2 Review Demonstration
**Course:** Community Project (Mini Project) — BCP308, III Semester  
**Institution:** Dayananda Sagar College of Engineering  
**Team:** Shourya (Lead), Pravar Singh, Shubham Pant, Shubham Sharma  
**Dataset:** UCI Beijing Multi-Site Air Quality (35,064 hourly records per station)  
**Guide:** Prof. Ensteih Silvia

## 1. Data Ingestion & Quality Control (Pravar Singh)
Loading the official UCI multi-site air quality dataset, constructing continuous chronological datetime index, and performing linear temporal interpolation to resolve sensor dropout.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

csv_path = "data/beijing_multisite/PRSA_Data_20130301-20170228/PRSA_Data_Aotizhongxin_20130301-20170228.csv"
raw_df = pd.read_csv(csv_path)
raw_df["timestamp"] = pd.to_datetime(raw_df[["year", "month", "day", "hour"]])
raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)

# Handle missing sensor readings with linear temporal interpolation
missing_before = raw_df["PM2.5"].isna().sum()
raw_df["PM2.5"] = raw_df["PM2.5"].interpolate(method="linear").bfill().ffill()
for col in ["TEMP", "PRES", "DEWP", "WSPM", "NO2", "SO2", "CO", "O3"]:
    if col in raw_df.columns:
        raw_df[col] = raw_df[col].interpolate(method="linear").bfill().ffill()

print(f"Ingested {len(raw_df):,} hourly records from {os.path.basename(csv_path)}.")
print(f"Successfully imputed {missing_before} missing PM2.5 sensor records.")

## 2. Feature Engineering & Strict Temporal Lags (Shubham Pant)
Extracting cyclical time encodings ($\\sin/\\cos$) and past 24h & 48h lags. Chronological train/test split to prevent data leakage.

In [2]:
raw_df["hour_sin"] = np.sin(2 * np.pi * raw_df["hour"] / 24.0)
raw_df["hour_cos"] = np.cos(2 * np.pi * raw_df["hour"] / 24.0)
raw_df["month_sin"] = np.sin(2 * np.pi * raw_df["month"] / 12.0)
raw_df["month_cos"] = np.cos(2 * np.pi * raw_df["month"] / 12.0)

# 24-hour and 48-hour lagged predictors
raw_df["PM2.5_lag24"] = raw_df["PM2.5"].shift(24)
raw_df["PM2.5_lag48"] = raw_df["PM2.5"].shift(48)
raw_df["PM2.5_roll_mean24"] = raw_df["PM2.5"].shift(24).rolling(window=24).mean()
raw_df["PM2.5_roll_std24"] = raw_df["PM2.5"].shift(24).rolling(window=24).std()
raw_df["TEMP_lag24"] = raw_df["TEMP"].shift(24)
raw_df["WSPM_lag24"] = raw_df["WSPM"].shift(24)

df_model = raw_df.dropna().copy().reset_index(drop=True)
feature_cols = [
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "PM2.5_lag24", "PM2.5_lag48", "PM2.5_roll_mean24", "PM2.5_roll_std24",
    "TEMP_lag24", "WSPM_lag24", "PRES", "DEWP", "NO2", "SO2"
]

X = df_model[feature_cols]
y = df_model["PM2.5"]
y_persistence = df_model["PM2.5_lag24"]

# Strict Chronological Split (80% Train, 20% Holdout Test)
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
y_persist_test = y_persistence.iloc[split_idx:]
test_timestamps = df_model["timestamp"].iloc[split_idx:]
print(f"Training set: {len(X_train):,} hours | Holdout Test set: {len(X_test):,} hours.")

## 3. Baseline vs. ML Model Evaluation (Shourya)
Comparing the Persistence Baseline against the trained Gradient Boosted Regressor.

In [3]:
# Persistence Baseline Metrics
rmse_base = np.sqrt(mean_squared_error(y_test, y_persist_test))
mae_base = mean_absolute_error(y_test, y_persist_test)
r2_base = r2_score(y_test, y_persist_test)

# Gradient Boosted Decision Tree Model
model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, max_depth=8, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse_ml = np.sqrt(mean_squared_error(y_test, y_pred))
mae_ml = mean_absolute_error(y_test, y_pred)
r2_ml = r2_score(y_test, y_pred)

results_df = pd.DataFrame({
    "Metric": ["RMSE (µg/m³)", "MAE (µg/m³)", "R² Score"],
    "Persistence Baseline (t-24)": [f"{rmse_base:.2f}", f"{mae_base:.2f}", f"{r2_base:.2f}"],
    "ML Model (GBDT)": [f"{rmse_ml:.2f}", f"{mae_ml:.2f}", f"{r2_ml:.2f}"],
    "Percentage Gain": [f"-{(rmse_base-rmse_ml)/rmse_base*100:.1f}%", f"-{(mae_base-mae_ml)/mae_base*100:.1f}%", f"+{r2_ml-r2_base:.2f}"]
})
results_df

## 4. Visualizing Forecast Performance over Test Horizon
Plotting a 120-hour window showing ground truth vs. persistence baseline vs. machine learning forecast.

In [4]:
plt.figure(figsize=(14, 5))
sample_hrs = 120
plt.plot(test_timestamps.iloc[-sample_hrs:], y_test.iloc[-sample_hrs:], label="Actual PM2.5", color="#2c3e50", lw=2)
plt.plot(test_timestamps.iloc[-sample_hrs:], y_pred[-sample_hrs:], label=f"ML Forecast (RMSE: {rmse_ml:.1f})", color="#2980b9", lw=2, linestyle="--")
plt.plot(test_timestamps.iloc[-sample_hrs:], y_persist_test.iloc[-sample_hrs:], label=f"Persistence Baseline (RMSE: {rmse_base:.1f})", color="#e74c3c", lw=1.2, linestyle=":")
plt.title("24-Hour Ahead PM2.5 Forecast: ML vs Persistence Baseline (120-Hour Window)", fontsize=13, fontweight="bold")
plt.xlabel("Date & Time")
plt.ylabel("PM2.5 (µg/m³)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 5. 24-Hour Community Advisory Engine (Shubham Sharma)
Translating predicted PM2.5 values into CPCB AQI health impact categories and identifying safe exposure windows.

In [5]:
def classify_cpcb_aqi(pm25):
    if pm25 <= 30: return "Good", "#2ecc71"
    elif pm25 <= 60: return "Satisfactory", "#a8e063"
    elif pm25 <= 90: return "Moderate", "#f39c12"
    elif pm25 <= 120: return "Poor", "#e67e22"
    elif pm25 <= 250: return "Very Poor", "#e74c3c"
    else: return "Severe", "#8e44ad"

next_24_times = [t.strftime("%H:%M") for t in test_timestamps.iloc[-24:]]
next_24_vals = [max(0.0, v) for v in y_pred[-24:]]
colors = [classify_cpcb_aqi(v)[1] for v in next_24_vals]

plt.figure(figsize=(12, 5))
bars = plt.bar(next_24_times, next_24_vals, color=colors, edgecolor="#2c3e50", alpha=0.85)
plt.axhline(60, color="#27ae60", linestyle="--", label="Satisfactory Cutoff (60 µg/m³)")
plt.axhline(120, color="#e67e22", linestyle="--", label="Moderate/Poor Cutoff (120 µg/m³)")
plt.title("Upcoming 24-Hour PM2.5 Forecast with Actionable Exposure Bands", fontsize=13, fontweight="bold")
plt.xlabel("Hour of Day")
plt.ylabel("Forecasted PM2.5 (µg/m³)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()